# 05a — pLLM zero-shot feature extraction (ESM scores)

**Purpose.** Extract zero-shot substitution scores from the ESM-family model ladder for all
4,783 single missense TEM-1 variants. This is the GPU half of the pLLM zero-shot baseline;
the local notebooks `03_EDA_pllm_zeroshot` and `04_pllm_zeroshot_benchmark` read the parquet
this notebook produces.

**Scores computed** (two schemes, per Meier et al. 2021):
- **masked-marginal** — mask the mutated site, score `log P(mut) − log P(wt)` from the masked
  distribution. Best of the four schemes in the source study (project decision D030).
- **wildtype-marginal** — same log-ratio read from a single unmasked forward pass over WT
  (D031). Within ~1% of masked-marginal at a fraction of the compute.

Both are *surprisal differences* at the mutated position (a log-likelihood ratio): a
substitution to a residue the model finds unlikely in context scores as more disruptive.

**Model ladder** (project decisions D010–D013):

| key | model | params | access |
|---|---|---|---|
| `esm1b` | ESM-1b | 650M | HF `facebook/esm1b_t33_650M_UR50S` |
| `esm1v` | ESM-1v (seed 1) | 650M | HF `facebook/esm1v_t33_650M_UR90S_1` |
| `esm2_150m` | ESM-2 | 150M | HF `facebook/esm2_t30_150M_UR50D` |
| `esm2_650m` | ESM-2 | 650M | HF `facebook/esm2_t33_650M_UR50D` |
| `esm2_3b` | ESM-2 | 3B | HF `facebook/esm2_t36_3B_UR50D` |
| `esmc_300m` | ESM-C | 300M | EvolutionaryScale SDK `esmc_300m` |
| `esmc_600m` | ESM-C | 600M | EvolutionaryScale SDK `esmc_600m` |

---
## GPU sizing — read before you pick a runtime

Scoring is cheap: masked-marginal needs **one forward pass per mutated position** (263 masked
passes + 1 WT pass per model), not one per variant. Wall time is a few minutes per model.
VRAM is the only constraint, and only the 3B model drives it (fp16):

| runtime | VRAM | what fits |
|---|---|---|
| **T4** (Colab free) | 16 GB | everything **except** ESM-2 3B is comfortable; 3B is tight and may OOM |
| **L4** (Colab Pro) | 24 GB | **the whole ladder incl. 3B — recommended, one session** |
| **A100** | 40/80 GB | whole ladder, fastest |

**Recommendation: one L4 session runs all seven models.** On a free T4 the six ≤650M models
run fine; if the 3B cell OOMs, either switch to L4 for that one model or skip it (the cell
catches the OOM, records the failure, and continues — the merge step drops missing columns).


In [ ]:
# 1. runtime + GPU check
import subprocess, sys
print(sys.version.split()[0])
try:
    out = subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"]).decode().strip()
    print("GPU:", out)
except Exception as e:
    print("NO GPU DETECTED — set Runtime > Change runtime type > GPU. (", e, ")")


In [ ]:
# 2. installs
#  - transformers/torch: ESM-1b, ESM-1v, ESM-2 (via EsmForMaskedLM). NOT fair-esm, so no
#    `import esm` namespace clash with the EvolutionaryScale SDK below.
#  - esm (EvolutionaryScale): ESM-C 300M/600M only. `import esm` == this SDK.
%pip -q install "transformers>=4.40" "torch" "pandas" "pyarrow" "esm>=3.1.1"
print("installs done")


In [ ]:
# 3. config + variant slice — SELF-CONTAINED (no upload needed)
# the variant list is embedded below as a gzip+base64 blob of seq_ids (e.g. "H24A" =
# wt_aa H, position 24, mut_aa A), so this notebook needs NO external files. if you'd
# rather point at your own copy, set SCORING_DATA_DIR and drop tem1_variants_for_scoring.parquet there.
import os, json, gzip, base64, re, time, gc, math
from pathlib import Path
import numpy as np, pandas as pd, torch

WT = "MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRVDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEPELNEAIPNDERDTTMPAAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWFIADKSGAGERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW"
_VARIANT_BLOB = "H4sIAPnoYmoC/yXdSZK1KhBA4Xmt5r9cFRhW2IdN2IVG7X8jz3Pe5BuSikBBmuUdQvH7M4SihgZa6KCHESaYYYUNdjjghAtueODvZwvlL9TQQAsd9DDACBPMsMAKOxxwwgU3PPD304bqFxroYYQJZtjhgAvunyvEX6ihgRY66GGAESaYYYEVNtjhgBNs9IG/nzmkX6ihhR4GGGGCBVbYYIcDTrjg/rlD/oUaGmihhxFmWGCFDXY44IQLHvj7mb7/fqGGBlrooIcBRphhgRU22OGAEy644YG/n/v7+YUaGmihgx4GGGGCGRZYYYMdDjjhgge4+sDVB64+cPWBqw9cfeDqA1cfuPrA1QeuPnD1gasPXH3g6gNXH7j6wNWHv5/m+/2FGlrooIcBRphghhU22OGAEy644e/n9/tOppcGWuighwFGmGCGFTbY4YATLrjhHe3fdzK91NBABz0MMMIEMyywwgY7HHDCBTc8QEdUdERFR1R0REVHVHRERUdUdERFR1R0REVHVHRERUdUdERFR1R0REVHVA/8/ezfd9K91NBACx30MMAIE8ywwAobHHDCBTc88E66L5Puy6T7pgZa6KCHAUaYYIEVNtjhgBMuuOGBv5/++07ElxoaaGGECWZYYIUNdjjghAtueOAdIMU7EV8aaKGDHgYYYYIZFlhhgx0OOOGCGx74+zmKdza+1NBACx30MMAIE8ywwAob7HDCBTc88E724p2NLzU00EIHPQwwwgQzLLDCBjsccMIFD7w9XrxT8qWGBlroYIQJZlhghQNOuOCGB/5+/or379tLDQ200EEPA4wwwQwLrLDBDgeccMENz89YvPPypYYGWuighwEmmGGBFTbY4YATLrjBlt9pX/BHrnjn5UsDHfQwwAgTzLDCDgeccMEN76Au3tn4UkMDLXTQwwAjTLDAChvscMAJF9zwTvbinYgvNbTQQQ8DjDDBDAussMEOB5xwwQ22/M7zInPhmQvPXHjmwjMXnrnwzIVnLjxz4ZkLz1x45sIzF5658MyFZy48c+HvbFzL92/jSw0NtNBBDwOMMMEMC2xwwAkX3PDA389ZvhPxpYYGWuhhhAlmWGGDHQ644IZ3VpTvpHupoYEWOhhghAlmWGCFDXY44IQLbnjg/QtbvpPupYYGWuighwFGmGGBFTbY4YATLrjhgb+fsXzn4EsNDbTQQQ8DTDDDAitscMAJF9zwwDssynf6vdTQQAsd9DDACBMssMIGOxxwwgU3PPBOv5LpVzL9SqZfyfQrmX4l069k+pVMv5LpV1YLrLDBDgeccMENDzBAIgMkMkAiAyQyQN7Z+MIoeWfjC0MlMlQiQ+Wdki+Ml8h4iYyXyHiJjJfIeHln48vfT1e+U/KlhgZa6GGECWZY4YATLrjhgfdvSvlOv5caGmihgx4GGGGCGRZYYYMdTrjghgfeo0L1zsGXGhpooYMeBhhhghkWWGGHA0644Ib3CVbvHHypoYEOehhghAlmWGCFDXY44IQLbniA5gPNB5oPNB9oPtB8oPlA84HmA80HWg40Gmg00Gig0UCj70Q8qncivtTQQAsd9DDACBPMsMAKG+xwwgU3PPCOiOqdiC81NNBCDwOMMMEMC+xwwAkX3PAAD49zXsU5r+KcV3HOqzjnVZzzKs55Fee8inNexTmv4pxXcc6rOOdVnPMqznkV57yKc17FOa965+BSvXPwpYYGWuighwFGmGCGFTbY4YATLrjhAZqPNB9pPtJ8pPlI85HmI81Hmo80H2k+0nyk+UjzkeYjzUeajzQfaf6dg2f1zsGXGhpooYMeBhhhghkWWGGDHQ644IYH/n6u6p2XLzU00EIHPQwwwgQzLLDCBjsccMIND7zjJb7z8qWGBlroYYARJphhgRU22OGAEy644YH3z058/yK+DDDCChsccMIF774zsu+M7Dsj+87IvjOy74zsOyNbzsiWM7LljOw2I7vNyG4zstuM7DYju834TrU5vlPtpYYGWuighwFGmGCBFTbY4YATLrjhAZovaL6g+YLmC5ovaLmg5YKWC1ouaLmg5YKWC1ouaLmg5YKW3/lWx3eWvTTQQgc9DDDCBDMssMIGOxxwwgU2+sC7wYjvLHupoYEWOhhghAlmWGCFDXY44IQLbnjgPfLEd4K9NNBCBz0MMMEMC6ywwQ4HnHDBDQ8wLBLDIjEsEsMiMSwSwyIxLN4J9sLYSIyNxNh4J9gLAyQxQBIDJDFAEgMkMUDeCfbCY2TfGdl3RvadkX1nZN8Z2XdG9p2RfWdk3xnZd0b2nZF9Z2TfGdl3RvadkX1nZN8ZM6PknWBneifYSw0NtNBBDwOMMMEMC6ywwQ4HXHDDA++fk8RRMHEUTBwFE0fBxFEwcRRMHAUTR8HEUTBxFEwcBRNHwcRRMHEUTBwFE0fBxFEwcRRMTMnElExMycSUTEzJxJRMTMnEUTAxLxPzMjEvE0fBxORMTM7E5ExMzsTkTEzOxFEwkZ1JZGcS2ZlEdiaRnUlkZxLZmUR2JpGdSWRn0jtDX1bYYIcDTrjghgfeMZnI0yTyNIk8TSJPk8jTJPI0iTxNIk+TyNOkd5q+rLDBDgeccMEN72RK71x9qaGBFjoYYIQJZlhghQ12OOCEC2544O9nT+9cfamhgRY66GGAESaYYYEVNjjghAtueODd06T3L+JLDQ100MMAI0wwwwIrbLDDASdccMMDXH3i6hNXn7j6xNUnrj5x9YmrT1x94uoTV5+4+sTVJ64+cfWJq09cfeLqE1fPhE1M2MSETUzYxIRNTNjEhE1M2MSETUzYxIRNTNjEhE1M2MSETUzYxIRNTNjEhE3vhO3zO2FfamighQ4GGGGCGRZYYYMdDjjhghseeCdsZsJmJmxmwmYmbGbCZiZsZsJmJmxmwmYmbGbCZiZsZsJmJmxmwmYmbGbCZiZs/tB8oPlA84HmA80Hmg80H2g+0Hyg+UDzgeYDzQeaDzQfaD7QfKD5QPPvhB3zO2FfamighQ56GGCCGRZYYYMdDjjhghse+PsZMq8qMq8qMq8qMq8qMq8qMq8qMq8qMq8qMq8q8jthX1bYYIcDTrjghgf+fv7yO2tfamighQ56GGCECWZYYIUNdjjghAtueBfk/M7alxoaaKGDHgYYYYIZFlhhgx0OuOAGW36nVSbPmsmzZvKsmTxrJs+aybNm8qyZPGsmz5rJs2byrJk8aybPmsmzZvKsmTxrJs+aybPmd9au+Z21LzU00EIHPQwwwgQzLLDBDgeccMEN7yqf3wn7UkMLHfQwwAgTzLDAChvscMAJF9zwwLsefP69MxZrbbTVTnsddNJVdz301Etvff9Gff7x2uO11kZb7bTXUWddddNDT730/zbfFfjzj2Pla62N9jrprIuuuumuh5566TsGP//eyYm1Ntpqp70OOuqksy666qa7Hnrqpf/Heofl5987X7HWRlvttNdBR5101lU33fXQS2/9P8p7Bv384xD6WmujrXba66CjTjrroqvueuipl9766B9PqvLpVz79yqdf+fQrn37l039nNToG3nmNjoR3ZqPjoXI8vJMbHRWVo6JyVFSOineGX59/vL98rbXRVjvtddBRJ5110VU33fXQU2991HGYHIfJcZgch++cR0fjO+txVEdmcmQmx2RyTCbHZHJMJsdkcky+8x8ffc+Cn3/vOoC1Ntpqr6POuuiqm+566KmX3vro+yfk83lXA6y10VY77XXUSWddddNdDz310lsfZeX5fGpttNVOex101EkXXXXTXQ899dJbH2VsfMKv1tpoq70OOuqksy666qa7HnrqrY++C/bnw778tdZWO+110FEnnXXRVTfd9dBTL7310XfT9fm8KwnW2mirg4466ayrbrrroadeeuujfz/L5/OuIVhro6122uugo04666qb7nroqZfe+qhPrfKpVT61yudVdepTq3xelc+r8nlVPq/Kp1P5dCqfTuXTqViXPvFXa2201U57HXTUSWdddNVNdz301EsffTeVn8+7SmCtjbbaaa+DjjrprIuuuumup15666OsS5/8q7U22mmvg4466ayLrrrroadeeuujzOLg/iG4fwjuH4L7h+D+Ibh/CO4fwr9RJ1101V0PPfXSWx/9+6k/4d1RYKOtdtrroKNOOuuiq26666GnXnrro3/vX88QfrXWRlvttNdBR5101kVX3fXQS2999D1vf8K7YmCjrfY66KiTzrroqpseeuqlt9p+YfuF7Re2z5n+QyUTGqUwSmGUwiiFUQqjFEbhcP+huAmNVRirMNa7Vowfipuw1kZb7bTXQSedddFVN9310FMvvfVR1orgWhFcKyh7wlY77XXQUSedddFVN9310FNvfZT1kHoprLXRVjvtddBRJ5111U13PfTUS2991NGYHI3J0ZgcjckRmByByRGYHIHJEZgce8mxlxx7ybGXHHvJscch40NFFdbaaqe9DjrqpLMuuuqmux566qW3PvoemD7UXGGtjbbaaa+DjjrprItuuuuhp15666OMEMqwsNZGW+2010FHnXTWRVfddNdDT731UWYcRVnYaq+Tzrropoeeeimz6eua8HVN+LomUISFtubK8HVl+LomfF0Nvq4GlGGhLbsmfF0TKMV6+6Sw/wv7v7D/C/u/sP8L+7+w/wv7v7D/C/u/sP8L+7+w/wv7v7D/C/u/+D+K/c+76A81Wlhro6122uugo0666Kqb7nroqZfe+qgRKyNWRqyMWBmxMmJlxMqIlRErI1ZGrIxYGbEyYmXEyoiVESsjVkasjBiNGI0YjRiNGI0YjRiNGI0YjRiNGI0YjRiNGI0YjRiNGI0YjRgd1clRnRzVyVGdHNXJUZ0c1clRnRzVyVGdHNXJUZ0c1clRnRzVyVGdHNXJUZ0c1cmI2YjZiNmI2YjZiNmI2YjZiNmI2YjZiNmI2YjZiNmI2YjZiNmImb8mxb9frbXRVjvtddBJZ1101U13PfTUS299lB01RWJYa6OtdjroqJPOuuiuh5566a2PGiUYJRglGCUYJRglGCUYJRglGCUYhQKWDxVjaMRgxGDEYMRgxGDEQB6A8jGstdFWO+110FEnnXXRVXc99NRLb32UcysFZlhro6122uugo8666Kqb7nroqZfe+ii7X2rOsNZGO+110FEnnXXRVTfd9dBTL731UWZ94TpTuM4UrjOF60zhOlO4zhSuM4XrTOE6U7jOFK4zhetM4TpTuM4UrjOF60zhOmOF2itzsDDXUZjrKMx1FOY6CnMdhbmOwlxHYa6jMNdRmOsozHUU5joKcx2FuY7CXEdhrqMw11GY6ygif8uoXcNGW+2010FHnXTWRVfddNdDT7301kf/froPNW1Ya6Ot9jroqJPOuuiqm+566KmX3vooz7H0LFN6lik9y5SeZUrPMqVnmdKzTOlZpvQsU3qWKT3LUPaGux566qW3Pkq+hRo4rLXRVjvtddRJZ111010PPfXSWx9lx0WRHNbaaKud9jroqJPOuumuh5566a3siqmVw1obbbXXQUeddNZVN9310FMvvfVR1s/SvEdp3qM071Ga96B4DgcdddJZF111010PPfXSWx9ln0xBHdbaaqe9DjrqrIuuuumuh5566a2OispRUTkqKkdF5aioHBWVo6JyVFSOispR8a4n6NioHBuVY6NybFSOjcqxUTk2KseGWZHSrEhpPqQ0H1KaCSnNgZTmQEpzIKXZj9LsR2n2ozT7UbpulO5GSncjpbuR0t1I6W6kdDdSuhsp3Y2U7kZKdyOlu5HS3UjpbqR0N1K6GynT/+3fSl7ForuPVXcfy+4+1t19rLn7WHT3seruY9ndx7q7j4V3HyvvPpbefay9+1h897H67mP53cf6u0/pylC5MlSuDJUrQ+XKULkyVK4MlStD5cpQuTJUrgyVK0PlylC5MlSuDJUrQ+XKULkyVK4MFOY1HyrzsNZWO+110FEnnXXRVTfd9dBTL731UfqTYj2stdFWO+110FEnnXXRVTfd9dRLb3307+f5UM6HtTbaaqe9DjrqpLMuuuqmux566qW3sjegwg8b7bTXQSdddNVNDz310lvZU1ng97HC72OJ38cav49Ffh+r/D6W+X2s8/tY6Pex0u9jqd/HWr+PxX4fq/0+lvt9rPf7WPD3seLvQ8nfG7fyLii8/VD1h95L5b1U3gvVtx+q/nBW76vyvirvixrcDwWA6N1V3l3l3VGI+6k811SeayrPNZXnmspzTeW5pvJcU3muqTzXVJ5rKs81leeaynNN5bmm8lxTea6pPNdUnmsqzzUUBq4fKgOx1kZb7bTXQUeddNZFN9310FMvvfVRe9WsaWXWtDJrWpk1rcyaVmZNK7OmlVnTyqxpZb6UYkG0P82aVmZNK7OmlVnTyqwphYO/H4oGsdFWO+110FEnnXXRVTfd9dBTL731Uc5Q8fOrtTbaaqe9DjrprIuuuumuh5566a2PMi+oSMRaG221014HHXXSWRdddddDT7301kcZM9QsYq2Nttppr4OOOumsi26666GnXnrro6zSlDFira122uugo04666KrbrrroadeeuujjNLo6SZ6uomebqKnm+jpJnq6iZ5uoqeb6OkmerqJnm6ip5vo6SZ6uomebqKnm+jpJnq6ofTx+FD7iLU22mqnvQ466qSzLrrqprueeumtj9qr0V6N9mq0V6O9Gu3VaK9GezXaq9FejfZqtFejvRrt1WivRns12qvRXo32qlmU6L4lum+J7lui+5boviW6b4nuW6L7lui+Jbpvie5bovuW6L4lum+J7lui+5boviWaRYlmUaJZlGgWJZpFiWZRolmUaBYlmkWJZlGiWZRoFiWaRYlmUaJZlGgWJZpFiWZRolmUaBaFWsrlQzEl1tpoq532Ouiok8666qa7Hnrqpbc+ygpAfSXW2mirnfY66KiTzrroqrseeuqltz7KuprMuFJuia122uugo04666KrbrrroadeeuujRjRDm8zQJjO0yQxtMkObzNAmM7TJdzfJPG3y3U0yW5vM1iaztclsbTJbm8zWJt/gUJaJPsfC51j4HAufY+FzLHyOhc+x8DkWPsfC51j4HAufY+FzLHyOhc+x8DkWPsfC51j4HAvvsfQeS++x9B5L7670vkrvq/S+Su+o9I5K76j0jkrvqPSOSu/ItzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJtzbJzEkyc5LMnCQzJ8nMSTJzksycJDMnycxJMnOSzJwkMyfJzEkyc5LMnCQzJ8kTUDJzksycUNA5f6joxFobbbXTXgcdddJFV91010NPvfTWR1mxkyej5MkoeTJKnowo9MReBx110lkXXXXTXU+99NZHyfJR94m1Ntpqp70OOuqsq26666GnXnrro/QnRaBYa6OtdtrroKNOuuiqm+566KmX3vqoEYMRgxGDEYMRgxGDEYMRgxGDEYMRgxGDsYKxgrGCsYKxgrEC45MiUWy01U57HXTUSWdddNVNDz311kfJnGQzJ9nMSTZzks2cZDMn2ZxJNmeSzZlkcybZnEk2Z5LNmWRzJtmcSTZnks2ZZHMm2R1LdseS3bFkdyzZvUp2r5Ldq2T3Ktm9SnaXkt2fZPcn2f1Jdn+S3Z9k9yfZ7Gs2+5rNvmazr9nsazb7ms2+ZrOv2exrNvuazb5ms6/Z7Gs2+5rNvmazr9nsazb7ms2+Zk9D2dNQ9jSUPQ1lT0PZ01D2NJQ9DWVPQ9nTUPY0lD0NZU9D2dNQ9jSUPQ1lT0PZ01D2NJTdn2T3J9n9SXZ/kt2fZPcn2f1Jdn+S3Z9k9yfZ/Ul2f5Ldn2T3J9n9SXZ/kt2fZPcn2f0JZadv3Ow9Zu8xe3fZu8veXfbusneXvbvsfWXvK3tf2fvK3lf2vt514zdQa4qNttppr4OOOumsi6666a6Hnnrp/1Ee/fs5A5WoWGujrXba66CjTjrrqpvueuiltz76rr2B2lSstdFWO+110FEnnXXRVTfd9dRLb33072cPVK5irY222mmvg4466ayLrrrpoadeeuujRiyMWBixMGJhxMKIhRELIxZGLIxYGLEwYmHEwoiFEQsjFkYsjFgYkXfHgfpVrLXRVjvtddBRJ11110NPvfTWR98zcqBmFWtttNVOex100lkXXXXTXQ899dJbH31POoGaVay11U57HXTUSWdddNVNdz301EtvffTv5wnUrGKtjbbaaa+DjjrprIuuuumuh5566a3vfjVQuYq1ttppr6NOOuuqux566qW3PsqHjqhZxVob7bTXQUeddNZFV91010NPvfTWdyccqFnFRlvttNdBR5101kVX3XTXQ0+99NZHGSFUrmKtrXba66CjTjrroqtuuuuhp15666PvXi5Qv4q1Ntpqp70OOuqsi6666a6HnnrprY/+/dyB+lWstdFWO+110FEnnXXVTXc99NRLH/UJlj7B0idY+gRLn2DpEyx9gqVPsPQJlj7B0idY+gRLn2DpEyx9gqVPsPQJlj7B0ifIv44Gqlix1kZbHXTUWRfddNdDT7301kf5jhgVrVhro6122uugo04666Kr7nroqZfe+iir8cdPeX38lhcVrdhqp70OOuqki6666a6HnnrprY8aMRsxGzEbMRsxGzEbMRsxGzEbMRsxGzEbMRsxGzEbMRsxGzEbMfN3nLpWrLXRVjvtddBRJ5110VU33fXUS299lF1KcJcS3KUEdynBXUpwlxLcpQR3KcFdSnCXEtylUOOKq26666GX3vooc4G6Vmy01U57HXTUSWdddNVNdz301EtvfZTnGPiv8kClKzbaaqe9DjrqpIuuuumuh5566a2PMjv4gB/W2mirnfY66KiTzrroqrseeuqltz5qr7rCBFeY4AoTXGGCK0xwhQmuMMEVJrjCBFeY4AoTXGGCK0xwhQmuMMEVJrjCBNeW4NoSXFuCa0twbaHqFQcdddJZF111010PPfXSWx9l/0DVK9baaKud9jroqJPOuuiqm+566KmX3vr30wWqXrHWRlvtddBRJ5110VU33fXQUy+99VH2gVTDYq2Nttppr4NOOuuiq26666GnXnrro4yZr2eir2eir2eir2eir2eir2eir2eir2eir2eir2eir2eir2eir2eir2eir2eir2eir2eir2ciqmGbQDUs1tpqp70OOuqksy666qa7Hnrqpbc+yr7C7xQGP1QY/FJh8FOFwW8VBj9WGPxaYfBzhcHvFQY/WBj8YmHwk4XBbxYGP1oY/Gph8LOFwe8Wvj7Kisr3C7HWRlvttNdBR5101lU33fXQS299lJlIPS3W2mirnY466ayLrrrproeeeumtjzpaXGG+rjBfV5ivK8zXFebrCvN1hfm6wnxdYb6uLV/Xlq9ry9e15eva8nVt+bq2fN29fF1hvq4wX1eYryvM1xXm6wrzdYX5usJ8XWG+rjBfV5ivK8zXFebrCvN1hfm6wnxdYb6uMF9XGGpo20ANLdbaaKe9DjrqpLMuuuqmux566qW3PsrfemposdZGW+2010FHnXTWRVfddNdTL731UXs126vZXs32arZXs72a7dVsr2Z7Ndur2V7N9mq2V7O9mu3VbK9mezXbq9le5UsagRparLXRVjvtddBRJ5110VU33fXQS299lF71Q4vBLy0GP7UY/NZi8GOLwa8tBj+3GPzeYvCDi8EvLgY/uRj85mLwo4vBry4GP7sY/O5i8MOLwS8vButpg/W0wXraYD1tsJ42WE8brKcN1tMG62mD9bTBetpgPW2wnjZYTxuspw3W0wbraYP1tMF62kA97Riop8VaG221014HnXTWRVfddNdDT7301keNWBixMGJhxMKIhRELIxZGLIxYGLEwYmHEwoiFEQsjFkYsjFgYsTCiq03halO42hSuNoWrTeFqU7jaFK42hatN4WpTuJ8pXHMK15zCNadwzSlccwrXnMI1p3DNoZ72tTJiZcTKiJURKyNWRqyMWBmxMmJlxMqIlRErI1ZGrIxYGbEyYmVEMrrBDz8Gv/wY/PRj8NuPwY8/Br/+GPz8Y/D7j8EPQAa/ABn8+mPw84/B7z8GPwAZ/AJk8BOQgUpadJQmR2lylCZHaXKUJkdpcpQmx2dyfCbHZ3J8Jsdncnwmx2dyfCbHZ3J8JsdnYvdLDS3W2mirnfY66KiTzrroqrseeuqltz7KvoIaWqy11U57HXTUSWdddNVNdz301EtvfZT+pIYWa2201U4HHXXSWRddddNdDz310lsfZSdTupMp3cmU7mRKdzKlO5nSnUzpTqZ0J1O6kyndyZTuZEp3MqU7mdKdTOlOpnQnU7qTKd3JUFW7Bapq0a+Ff/1c+NfvhX/9YPjXL4Z//WT412+Gf/1o+Nevhn/9YvjXT4Z//Wb414+Gf/1q+NfPhn/9bviXvxRU1WKtjbbaaa+DjjrprIuuuumuh15666P8paCqFmtttNVOex101ElnXXTVTXc99dJbH2UVLc30lmZ6SzO9pZne0kxvaaa3NNNbmuktzfSWZnpLM72lmd7STG9pprc001ua6S3N9JZmeq2wDVbYBr5miY222mmvg4466ayLrrrproeeeumjRkxGTMZKRklGSbafbC3ZQrKFZAuegEpPQKUnoNITUOkJqPQEVHoCKj0BlZ6ASk9ApSeg0hNQ6Qmo9ARUegIqPQGVnoBKT0ClJyAqbP8CFbZYa6OtdtrroKNOOuuqm+566KmX3srn6KmtxUZb7XXQUSedddFVdz301Fv/b/kPg+3ztYFAJS0aJXRqrGCsYKxgrGCsYKxgLL45EKikRSMGIwYjBiPyVehAJS022mqng4466ayLbrrroadeeivztHJNqFwTKteEyjWhck2oXBMqV4PK1aByNahcDSpXg8rVoHI1qFwNKleDytWgKv6Pwtsl6mmx1kZb7bTXQUeddNZFV9300FMvvfVR9huV+43K/UblfqNyv1G536jcb1TuNyr3G5X7jcr9RuV+o3K/UbnfqNxvVO43KvcblfuNyv0G9bTvU/O3Fip/bKHy1xYqf26h8vcWKn9wofIXFyp/cqHyNxcqf3Sh8lcXKn92ofJ3Fyp/eKHylxcqf3qh8rcXKn98gXraJVBPi7U22mqnvQ466qSzrrrproeeeumtj7IToJ4Wa221014HHXXSWRddddNdDz310lsf5eQYfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRMUfRNEVS3ylyt6xomecaJnnOgZJ3rGiZ5xomec6BknerqJnmiiJ5roiSZ6oomeaKInGmpo10ANLdbaaKud9jroqJPOuuimux566qW3Puodfb2jr/fy9V6+3sXX6+fDuoGKWfT6v17/1+v/ev2+I46+I46+HY6+HY6+F46+F46+F46+F46+F46+C46+C46+C46+C46+Bab2dQzUvmKtjbbaaa+DTjrroqtuuuuhp15666OsDNGVIboyRFeG6MoQXRmiK0N0ZYiuDNGVIbomRNeE6JoQXROia0J0TYiuCdGMRzTjEc14RDMe0YxHNOMRzXhEMx7RjEc04xHNeEQzHtGMRzTjEc14RDMe0YxHNOMRzXhQ7/reXbInkz2Z7MlkTyZ7MtmTyZ5M9mSyJ5M9mezJZE8mezLZk8meTPZksieTPWnGI5rxiGY8ohmPaMYjmvGI5jqiuY5oriOa64jmOqJZjmiWI5rliGY5olmOaJYjmkdN5lGTedRkHjWZR03mUZN51GQeNZlHTeZRk3nUZB41mUdN5lGTedRkHjWZR03mUZN51ORbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+Rbm+RbG2pc58BnRdFf0qEaLVDpip36uzrBH9YJ/rJO8Kd1gr+tE/xxneCv6wR/Xif4+zrBH9gJ/sIONWmBGldktCSzHMksRzLLkcxyJLMcySxHMsuRzHIksxzJLEcyy5HMciSzHMksRzLLkcxyJLMcySwHNa5ToMYVa2201V4HHXXWRVfddNdDT7301kf/fobA50ix1kZb7bTXUSedddFVN9310FMvvZW3J1S6YqOtdtrroKNOOuuiq26666GnXnrr33/xHZ07uWoAAA=="

def _load_variants():
    # optional override: a local/Drive parquet takes precedence if present
    ext = Path(os.environ.get("SCORING_DATA_DIR","")) / "tem1_variants_for_scoring.parquet"
    if os.environ.get("SCORING_DATA_DIR") and ext.exists():
        print("using external slice:", ext); return pd.read_parquet(ext)
    ids = gzip.decompress(base64.b64decode(_VARIANT_BLOB)).decode().split("\n")
    rows=[]
    for s in ids:
        m=re.fullmatch(r"([A-Z])(\d+)([A-Z])", s)
        rows.append({"seq_id":s,"wt_aa":m.group(1),"position":int(m.group(2)),"mut_aa":m.group(3)})
    return pd.DataFrame(rows)[["seq_id","position","wt_aa","mut_aa"]].sort_values(
        ["position","mut_aa"]).reset_index(drop=True)

variants = _load_variants()
L = len(WT)

# boundary check (CLAUDE.md): 0-based seq index = position-1; WT residue there must equal wt_aa.
seqidx = variants.position.values - 1
assert (seqidx>=0).all() and (seqidx<L).all(), "position out of range for WT"
assert all(WT[i]==aa for i,aa in zip(seqidx, variants.wt_aa.values)), "wt_aa != WT[pos-1]"
assert variants.seq_id.is_unique, "duplicate seq_id"

AA = list("ACDEFGHIKLMNPQRSTVWY")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path("out"); OUT.mkdir(exist_ok=True)
print(f"variants={len(variants)} positions={variants.position.nunique()} L={L} device={DEVICE}")
print("unique mutated positions (0-based):", len(np.unique(seqidx)))


In [ ]:
# 4. HF ESM-1/2 scorer (masked-marginal + wildtype-marginal)
# ESM HF tokenizers prepend CLS and append EOS, so sequence index i maps to token index i+1.
from transformers import AutoTokenizer, EsmForMaskedLM

@torch.no_grad()
def score_hf(model_id, dtype=torch.float16, mask_batch=16):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = EsmForMaskedLM.from_pretrained(model_id, torch_dtype=dtype).to(DEVICE).eval()
    aa_ids = torch.tensor([tok.convert_tokens_to_ids(a) for a in AA], device=DEVICE)
    assert (aa_ids>=0).all() and tok.unk_token_id not in aa_ids.tolist(), "AA not in vocab"
    mask_id = tok.mask_token_id

    enc = tok(WT, return_tensors="pt").to(DEVICE)     # [1, L+2]
    ids = enc["input_ids"]
    # sanity: token at index i+1 decodes to WT[i]
    assert tok.convert_ids_to_tokens(int(ids[0,1]))==WT[0]

    positions = np.unique(seqidx)                     # 0-based seq positions to score
    tok_idx = positions + 1                           # -> token indices

    # wildtype-marginal: one unmasked forward
    wt_logits = model(**enc).logits[0].float()        # [L+2, V]
    wt_lsm = torch.log_softmax(wt_logits, -1)         # log P(. | WT) per token slot
    # per scored position, log-prob of each of the 20 AAs
    wm_logp = wt_lsm[tok_idx][:, aa_ids].cpu().numpy()          # [P, 20]

    # masked-marginal: mask each position (batched), read that slot's distribution
    mm_logp = np.empty((len(positions), 20), dtype=np.float32)
    base = ids.repeat(len(positions), 1)              # [P, L+2]
    for a in range(len(positions)):
        base[a, tok_idx[a]] = mask_id
    for s in range(0, len(positions), mask_batch):
        b = base[s:s+mask_batch]
        lg = model(input_ids=b, attention_mask=torch.ones_like(b)).logits.float()  # [b, L+2, V]
        lsm = torch.log_softmax(lg, -1)
        rows = torch.arange(b.shape[0])
        slot = torch.tensor(tok_idx[s:s+mask_batch], device=DEVICE)
        mm_logp[s:s+mask_batch] = lsm[rows, slot][:, aa_ids].cpu().numpy()

    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return positions, np.array(AA), wm_logp, mm_logp

print("HF scorer ready")


In [ ]:
# 5. ESM-C scorer (EvolutionaryScale SDK) — tensor-level masking mirrors the HF path
# NB: mask by overwriting the token id in the encoded tensor (NOT by putting "_" in the string).
# ESM-C is BOS/EOS-bracketed like HF, so seq index i -> token index i+1.
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, ESMProteinTensor, LogitsConfig

@torch.no_grad()
def score_esmc(model_key):
    client = ESMC.from_pretrained(model_key).to(DEVICE).eval()
    tok = client.tokenizer                                   # EsmSequenceTokenizer (HF-style)
    aa_ids = torch.tensor([tok.convert_tokens_to_ids(a) for a in AA], device=DEVICE)
    unk = getattr(tok, "unk_token_id", None)
    assert (aa_ids >= 0).all() and (unk is None or unk not in aa_ids.tolist()), \
        "an AA token is missing/unk in the ESM-C vocab"
    mask_id = tok.mask_token_id; assert mask_id is not None

    base = client.encode(ESMProtein(sequence=WT)).sequence.to(DEVICE)   # token ids [L+2]
    assert base.ndim == 1 and base.shape[0] == L + 2, \
        f"ESM-C encoded length {tuple(base.shape)} != expected {L+2} (BOS/EOS bracket?)"

    def _seq_logits(tokens):
        out = client.logits(ESMProteinTensor(sequence=tokens), LogitsConfig(sequence=True))
        lg = out.logits.sequence                             # [1, L+2, V]
        assert lg.shape[1] == L + 2, f"ESM-C logits len {lg.shape[1]} != {L+2}"
        return lg[0].float()

    positions = np.unique(seqidx); tok_idx = positions + 1
    wt_lsm = torch.log_softmax(_seq_logits(base), -1)        # fails fast here if API path is wrong
    wm_logp = wt_lsm[tok_idx][:, aa_ids].cpu().numpy()

    mm_logp = np.empty((len(positions), 20), dtype=np.float32)
    for a in range(len(positions)):
        t = base.clone(); t[tok_idx[a]] = mask_id
        lsm = torch.log_softmax(_seq_logits(t), -1)
        mm_logp[a] = lsm[tok_idx[a]][aa_ids].cpu().numpy()

    del client; gc.collect(); torch.cuda.empty_cache()
    return positions, np.array(AA), wm_logp, mm_logp

print("ESM-C scorer ready")


In [ ]:
# 6. assemble per-variant scores from per-position log-prob tables
# score(scheme) = log P(mut) - log P(wt) at the mutated site (a surprisal difference).
def variant_scores(positions, aa_arr, wm_logp, mm_logp):
    pos2row = {int(p): r for r, p in enumerate(positions)}
    aa2col  = {a: c for c, a in enumerate(aa_arr.tolist())}
    wm = np.empty(len(variants)); mm = np.empty(len(variants))
    for k, (si, wa, ma) in enumerate(zip(seqidx, variants.wt_aa.values, variants.mut_aa.values)):
        r = pos2row[int(si)]; cw = aa2col[wa]; cm = aa2col[ma]
        wm[k] = wm_logp[r, cm] - wm_logp[r, cw]
        mm[k] = mm_logp[r, cm] - mm_logp[r, cw]
    return wm, mm
print("assembler ready")


In [ ]:
# 7. run the ladder — one model at a time, checkpoint each to out/ so a disconnect
# mid-3B never loses the small models. re-running skips models already on disk.
LADDER = [
    ("esm1b",     "hf",   "facebook/esm1b_t33_650M_UR50S"),
    ("esm1v",     "hf",   "facebook/esm1v_t33_650M_UR90S_1"),
    ("esm2_150m", "hf",   "facebook/esm2_t30_150M_UR50D"),
    ("esm2_650m", "hf",   "facebook/esm2_t33_650M_UR50D"),
    ("esm2_3b",   "hf",   "facebook/esm2_t36_3B_UR50D"),
    ("esmc_300m", "esmc", "esmc_300m"),
    ("esmc_600m", "esmc", "esmc_600m"),
]
# smaller masked-batch for the 3B model to stay under T4/L4 VRAM
MASK_BATCH = {"esm2_3b": 4, "esm2_650m": 12, "esm1b": 12, "esm1v": 12}

status = {}
for key, kind, ref in LADDER:
    ck = OUT/f"scores_{key}.parquet"
    if ck.exists():
        print(f"[skip] {key} — checkpoint exists"); status[key]="cached"; continue
    print(f"[run ] {key} ({ref}) ...", flush=True); t0=time.time()
    try:
        if kind=="hf":
            res = score_hf(ref, mask_batch=MASK_BATCH.get(key, 16))
        else:
            res = score_esmc(ref)
        wm, mm = variant_scores(*res)
        pd.DataFrame({"seq_id": variants.seq_id.values,
                      f"{key}__wildtype_marginal": wm,
                      f"{key}__masked_marginal":   mm}).to_parquet(ck, index=False)
        status[key]=f"ok {time.time()-t0:.0f}s"
        print(f"[done] {key} in {time.time()-t0:.0f}s  mm[min,med,max]="
              f"[{mm.min():.2f},{np.median(mm):.2f},{mm.max():.2f}]")
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        status[key]="OOM — needs a bigger GPU (L4/A100); skipped"
        print(f"[OOM ] {key}: out of memory — switch to L4/A100 for this model, or skip it")
    except Exception as e:
        status[key]=f"ERROR: {e}"
        print(f"[fail] {key}: {e}")
print("\nstatus:", json.dumps(status, indent=0))


In [ ]:
# 8. merge per-model checkpoints -> one tidy scores parquet keyed by seq_id
parts = sorted(OUT.glob("scores_*.parquet"))
assert parts, "no per-model checkpoints found — did any model run?"
merged = variants[["seq_id"]].copy()
for p in parts:
    merged = merged.merge(pd.read_parquet(p), on="seq_id", how="left")
# validate: seq_id count + alignment, and no all-NaN score columns
assert len(merged)==len(variants) and merged.seq_id.is_unique
score_cols = [c for c in merged.columns if c!="seq_id"]
bad = [c for c in score_cols if merged[c].isna().all()]
assert not bad, f"all-NaN score columns: {bad}"
merged.attrs = {}
FINAL = OUT/"pllm_zeroshot_scores.parquet"
merged.to_parquet(FINAL, index=False)
print("wrote", FINAL, merged.shape)
print("columns:", score_cols)
print(merged.head(3).to_string())


In [ ]:
# 9. quick self-check — a real scorer ranks WT-preferred substitutions above disruptive ones,
# so scores should correlate across schemes/models and be mostly negative (mut less likely than wt).
import itertools
mmc = [c for c in score_cols if c.endswith("masked_marginal")]
if len(mmc)>=2:
    corr = merged[mmc].corr(method="spearman")
    print("Spearman corr between models (masked-marginal):"); print(corr.round(3).to_string())
print("\nfraction of scores < 0 (mut less likely than wt) per column:")
print((merged[score_cols] < 0).mean().round(3).to_string())


## 10. Drop the output into the local pipeline

Download **`out/pllm_zeroshot_scores.parquet`** and place it at the exact path below so the
local notebooks (`03_EDA_pllm_zeroshot`, `04_pllm_zeroshot_benchmark`) pick it up on their next
run. This path matches the project's `paths.py` (`FEATURES_PLM_MM`):

```
1 - ML/data/features/plm_masked_marginal/pllm_zeroshot_scores.parquet
```

Nothing else needs to move — the local notebooks resolve everything else from `paths.py`.
Both schemes (masked-marginal and wildtype-marginal) for every model live in this one file,
one column per `{model}__{scheme}`.

In [ ]:
# 11. download helper (Colab)
try:
    from google.colab import files
    files.download(str(OUT/"pllm_zeroshot_scores.parquet"))
except Exception as e:
    print("not on Colab or download unavailable — grab out/pllm_zeroshot_scores.parquet manually.", e)
